In [ ]:
# %pip install --upgrade pip
# %pip install transformers==4.40.1 peft==0.4.0
# %pip install sentencepiece
# %pip install accelerate
# %pip install torch
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
# %pip install peft
# %pip install datasets
# %pip install bitsandbytes
# %pip install huggingface_hub
# %pip install ipywidgets

# import os
# from huggingface_hub import login
# # from .autonotebook import tqdm as notebook_tqdm

# login(token=os.getenv("HF_TOKEN"))


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = AutoModelForCausalLM.from_pretrained(
    'meta-llama/Llama-2-7b-chat-hf',
    trust_remote_code=True,
    device_map="cuda:0",
    torch_dtype=torch.float16,   # optional if you have enough VRAM
)
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-2-7b-chat-hf')

model = PeftModel.from_pretrained(base_model, 'FinGPT/fingpt-forecaster_dow30_llama2-7b_lora')
model = model.eval()


c:\_Coding\LLM-Predict-Stock\.venv\Lib\site-packages\huggingface_hub\file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

: 

In [5]:
# Make prompts
prompt = [
# '''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
# Input: 川普關稅政策帶來市場亂流，此外，密西根大學調查顯示，2月消費者預期未來1年通膨為4.3%，創2023年11月以來新高
# Answer: ''',
'''
What is the sentiment of this news

[Cryptocurrency Introduction]: Bitcoin (BTC) is a cryptocurrency launched in 2010. Users are able to generate BTC through the process of mining. Bitcoin has a current supply of 19,599,987. The last known price of Bitcoin is 42,945.87509791 USD and is up 0.58 over the last 24 hours. It is currently trading on 10709 active market(s) with $21,150,003,121.31 traded over the last 24 hours. More information can be found at https://bitcoin.org/.. It has a market capilization of 845807222784.

From 2023-11-12 to 2023-11-19, BTC-USD's stock price increased from 37054.52 to 37386.55. Company news during this period are listed below:

[Headline]: Bitcoin, BlackRock, And The End Of Decentralization
[Summary]: BlackRock's entry into cryptocurrency marks the end of the idea of a decentralized world currency. Learn more about Bitcoin's rally in 2023.
'''
]

# Set device
tokenizer.pad_token = tokenizer.eos_token


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)
tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512).to(device)
res = model.generate(**tokens, max_length=512)
res_sentences = [tokenizer.decode(i) for i in res]
print(res_sentences)
out_text = [o.split("Answer: ")[1] for o in res_sentences]


# Show results
for sentiment in out_text:
    print(sentiment)

cuda
["<s> \nWhat is the sentiment of this news\n\n[Cryptocurrency Introduction]: Bitcoin (BTC) is a cryptocurrency launched in 2010. Users are able to generate BTC through the process of mining. Bitcoin has a current supply of 19,599,987. The last known price of Bitcoin is 42,945.87509791 USD and is up 0.58 over the last 24 hours. It is currently trading on 10709 active market(s) with $21,150,003,121.31 traded over the last 24 hours. More information can be found at https://bitcoin.org/.. It has a market capilization of 845807222784.\n\nFrom 2023-11-12 to 2023-11-19, BTC-USD's stock price increased from 37054.52 to 37386.55. Company news during this period are listed below:\n\n[Headline]: Bitcoin, BlackRock, And The End Of Decentralization\n[Summary]: BlackRock's entry into cryptocurrency marks the end of the idea of a decentralized world currency. Learn more about Bitcoin's rally in 2023.\n</s>"]


IndexError: list index out of range